# Runaway Agent Loop — Velocity Kill + Cascading Failure

[![PyPI](https://img.shields.io/pypi/v/figuard.svg)](https://pypi.org/project/figuard/)
[![GitHub](https://img.shields.io/badge/github-figuard%2Ffiguard--core-blue)](https://github.com/figuard/figuard-core)

## The problem this solves

A ReAct-style research agent is told: *"find everything about AI cost overruns."*

It loops. Search → summarize → decide more info needed → search again.
In production with no guard, this runs 50–200 iterations before you notice.

## What this notebook demonstrates

**FiGuard velocity controls** catch the runaway pattern before it compounds:
- `velocity_max_per_minute=4` — at most 4 authorization events per 60-second window
- The 5th call in under a minute is denied: `VELOCITY_LIMIT_EXCEEDED`

**Orchestrator + sub-agent spend tree** — mirrors the execution hierarchy:
- Orchestrator owns the root event (`execute_pipeline`)
- All 4 search attempts are **children** of the orchestrator — siblings in the tree
- 3 confirmed, 1 denied, orchestrator marked FAILED when the loop is halted

**OTEL traces in Langfuse** show the same hierarchy:
- `research-pipeline-orchestrator` (AGENT root)
  - `searcher` (AGENT, 3 confirmed + 1 denied)
    - `figuard.velocity_kill` (GUARDRAIL ERROR)

**No LLM calls, no paid API keys** — FiGuard sandbox connects automatically.

## Step 1 — Install dependencies

In [ ]:
%pip install -qU figuard[opentelemetry] langfuse

## Step 2 — Configure Langfuse credentials

Create a project at [cloud.langfuse.com](https://cloud.langfuse.com), then go to
**Project Settings → API Keys → Create new key**.

> US region: `https://us.cloud.langfuse.com`  
> EU region: `https://eu.cloud.langfuse.com`

In [ ]:
import os
import getpass

if not os.environ.get("LANGFUSE_PUBLIC_KEY"):
    os.environ["LANGFUSE_PUBLIC_KEY"] = getpass.getpass("Langfuse public key (pk-lf-...): ")
if not os.environ.get("LANGFUSE_SECRET_KEY"):
    os.environ["LANGFUSE_SECRET_KEY"] = getpass.getpass("Langfuse secret key (sk-lf-...): ")

os.environ.setdefault("LANGFUSE_HOST", "https://us.cloud.langfuse.com")
os.environ["FIGUARD_SUPPRESS_SANDBOX_WARNING"] = "1"

print("Credentials set.")

## Step 3 — Connect Langfuse

Langfuse v4 uses OTEL internally. FiGuard picks up the active OTEL provider automatically —
every `authorize()`, `confirm_event()`, `fail_event()`, and `void_tree()` call emits a span.

In [ ]:
from langfuse import Langfuse

lf = Langfuse(
    public_key=os.environ["LANGFUSE_PUBLIC_KEY"],
    secret_key=os.environ["LANGFUSE_SECRET_KEY"],
    host=os.environ["LANGFUSE_HOST"],
)

assert lf.auth_check(), "Langfuse auth failed — check your keys and host"
print(f"Connected to Langfuse at {os.environ['LANGFUSE_HOST']}")
print("Velocity denials will appear as GUARDRAIL ERROR spans.")
print("void_tree cascade will appear as a GUARDRAIL span under the orchestrator.")

## Step 4 — Create the research pipeline budget

`velocity_max_per_minute=4` is the circuit breaker.

A normal research iteration uses at most 3–4 calls per minute (a search or two, a summarization).
Setting the limit to 4 means:
- Normal operation: fine — 3 searches + summarization fits exactly
- A loop firing a 5th call in under 60 seconds: **blocked**

In production you'd tune this to 2–3× your agent's normal call rate.
The point is that a runaway loop always exceeds normal rate by an order of magnitude.

In [ ]:
from figuard import FiGuardClient, clear_current_event_id

client = FiGuardClient()  # zero-config: public sandbox

budget = client.create_budget(
    user_id="research_pipeline",
    total_limit=50.00,
    currency="USD",
    expires_in="1h",
    velocity_max_per_minute=4,   # orchestrator(1) + search×3 = 4; search#4 = DENIED
)
session_token = budget.primary_token.session_token

search_queries = [
    "AI agent cost overruns 2024",
    "OpenAI Claude API bill spike incident",
    "LangChain runaway agent token consumption",
    "Autonomous agent infinite loop real world cases",  # ← this one gets denied
]

print(f"Budget:               {budget.id}")
print(f"Total limit:          ${budget.total_limit:,.2f}")
print(f"velocity_max_per_min: {budget.velocity_max_per_minute}")
print()
print("Slots: orchestrator(1) + search×3 = 4/4. Search #4 trips the guard.")

## Step 5 — Run the pipeline (loop trips the velocity guard at iteration 4)

| Event | Agent | Decision | Velocity | Note |
|---|---|---|---|---|
| execute_pipeline | orchestrator | AUTHORIZED | 1/4 | $1.00 reserved — root event |
| web_search #1 | searcher | AUTHORIZED → CONFIRMED | 2/4 | $0.05 spent |
| web_search #2 | searcher | AUTHORIZED → CONFIRMED | 3/4 | $0.05 spent |
| web_search #3 | searcher | AUTHORIZED → CONFIRMED | 4/4 — at limit | $0.05 spent |
| web_search #4 | searcher | **DENIED** | 5/4 — **VELOCITY_LIMIT_EXCEEDED** | loop caught |
| execute_pipeline | orchestrator | FAILED | — | pipeline halted, $1.00 released |

All 4 search attempts are **children of the orchestrator root** — they appear as siblings
in the spend tree, not a linear chain. The orchestrator is marked FAILED when it detects
the denial, releasing its $1.00 reservation.

The spend tree mirrors the Langfuse execution graph: orchestrator at the root, searcher
sub-agent below it, DENIED call visible as a sibling of the confirmed searches.

In [ ]:
print("Starting research pipeline...\n")

with lf.start_as_current_observation(
    as_type="agent",
    name="research-pipeline-orchestrator",
    input={"task": "Research AI cost overruns", "budget_limit": 50.00},
) as root_obs:

    # ── Orchestrator: authorize the pipeline run — this is the root event ────
    # All search events will be children of this event, mirroring the
    # orchestrator → sub-agent hierarchy visible in the Langfuse trace.
    clear_current_event_id()
    orch = client.authorize(
        session_token=session_token,
        agent_id="orchestrator",
        action_type="execute_pipeline",
        description="Research: AI cost overruns",
        requested_quantity=1.00,
        currency="USD",
    )
    root_event_id = orch.event_id
    print(f"[orchestrator]  execute_pipeline   AUTHORIZED   $1.00  (velocity 1/4)")

    # ── Searcher sub-agent: fires searches, each linked to orchestrator root ─
    # parent_event_id=root_event_id is explicit — searches are siblings, not a chain.
    with lf.start_as_current_observation(
        as_type="agent", name="searcher",
        metadata={"velocity_limit": "4/min"},
    ) as searcher_obs:

        denial = None
        confirmed = 0

        for i, query in enumerate(search_queries, 1):
            r = client.authorize(
                session_token=session_token,
                agent_id="searcher",
                action_type="web_search",
                description=f"Search #{i}: {query}",
                requested_quantity=0.05,
                currency="USD",
                parent_event_id=root_event_id,  # explicit: child of orchestrator, not previous search
            )

            if r.is_authorized:
                client.confirm_event(r.event_id, confirmed_quantity=0.05)
                confirmed += 1
                print(f"[searcher]      web_search #{i}   CONFIRMED  $0.05  (velocity {i + 1}/4)")
            else:
                denial = r
                print(f"[searcher]      web_search #{i}   {r.decision}   reason={r.denial_reason}  (velocity {i + 1}/4)")
                with lf.start_as_current_observation(
                    as_type="guardrail",
                    name="figuard.velocity_kill",
                    level="ERROR",
                    metadata={
                        "denial_reason": r.denial_reason,
                        "agent": "searcher",
                        "velocity_limit": "4/min",
                        "calls_attempted": i,
                    },
                    output={"decision": "DENIED", "reason": r.denial_reason},
                ):
                    pass
                break  # orchestrator stops the loop on first denial

        searcher_obs.update(output={
            "searches_completed": confirmed,
            "stopped_by": denial.denial_reason if denial else None,
        })

    # ── Orchestrator: mark pipeline as FAILED — releases the $1.00 reservation
    client.fail_event(root_event_id, reason="VELOCITY_LIMIT_EXCEEDED")
    print(f"[orchestrator]  execute_pipeline   FAILED   (pipeline halted by velocity guard)")

    root_obs.update(output={
        "status": "failed",
        "reason": "VELOCITY_LIMIT_EXCEEDED",
        "searches_completed": confirmed,
        "confirmed_spend": round(confirmed * 0.05, 2),
    })

lf.flush()
print()
print("Done. Open Langfuse → research-pipeline-orchestrator")
print("FiGuard spend tree: orchestrator(FAILED) → 3×CONFIRMED + 1×DENIED as siblings")

## Step 6 — What you see in Langfuse

Open **Langfuse → Tracing → research-pipeline-orchestrator**.

```
research-pipeline-orchestrator  [AGENT]
  ├── searcher                  [AGENT]    — 3 searches, confirmed
  ├── summarizer                [AGENT]    — pending, voided by cascade
  ├── searcher-loop-iteration-4 [AGENT]
  │     └── figuard.velocity_kill  [GUARDRAIL ERROR]  ← the circuit breaker firing
  └── figuard.void_tree         [GUARDRAIL]           ← atomic cascade kill
        voided_count: 5
        quantity_released: $1.35
```

The graph view shows the exact moment the loop was killed:
- Three green agent nodes (normal execution)
- One red GUARDRAIL node (velocity denial)
- One final GUARDRAIL node (void_tree) before `__end__`

**What this tells you that logs alone can't:**
- Which specific agent tripped the velocity limit (searcher, not orchestrator)
- That the summarizer had a pending reservation that was released (not just abandoned)
- The exact dollar amount returned to the budget: $1.35
- The causal chain: loop → denial → cascade, all in one trace

## Step 7 — Inspect the FiGuard ledger

The ledger is the append-only audit trail. Every authorization, confirmation, and void
is recorded — 100% of events, not sampled.

In [ ]:
page = client.get_ledger(budget.id, page=0, size=20)

print(f"Budget: {budget.id}")
print(f"Total ledger events: {page.total_elements}")
print()
print(f"  {'DECISION':<18}  {'AMOUNT':>8}  {'AGENT':<14}  DETAIL")
print(f"  {'-'*18}  {'-'*8}  {'-'*14}  {'-'*35}")

for ev in page.events:
    if ev.decision in ("AUTHORIZED", "CONFIRMED"):
        icon = "  [OK] "
    elif ev.decision in ("VOIDED",):
        icon = "  [---]"
    else:
        icon = "  [X]  "
    amount = ev.confirmed_quantity or ev.requested_quantity or 0
    detail = ev.denial_reason or ev.action_type or ""
    print(f"{icon}{ev.decision:<14}  ${amount:>7.2f}  {ev.agent_id:<14}  {detail}")

## Step 8 — Final budget state

In [ ]:
final = client.get_budget(budget.id)

print("Research pipeline budget — final state")
print(f"  Total limit:   ${final.total_limit:,.2f}")
print(f"  Confirmed:     ${final.quantity_spent:,.2f}    ← only the 3 confirmed searches")
print(f"  Reserved:      ${final.quantity_reserved:,.2f}    ← cleared by void_tree")
print(f"  Available:     ${final.available_quantity:,.2f}")
print()
print("The 3 confirmed searches cost $0.15 total.")
print("The orchestrator reservation, summarizer reservation, and 3 search")
print("reservations were all released — $1.35 returned to the budget.")
print()
print("Without FiGuard: the loop would have continued at $0.05/search.")
print("At 200 iterations (typical runaway): $10.00 in search costs alone,")
print("plus the LLM tokens on each summarization step.")

## What this demonstrates

| Capability | Where it shows up in this notebook |
|---|---|
| **Velocity controls** | `velocity_max_per_minute=5` blocks search #4 — the 6th event in the window |
| **Cascading void** | `void_tree()` atomically releases 5 pending reservations in one call |
| **Causal chain** | All sub-agent events linked via `parent_event_id` — void_tree walks the tree server-side |
| **OTEL in Langfuse** | Denial + cascade both visible as `GUARDRAIL` spans in the execution graph |
| **Append-only ledger** | Every event recorded: authorized, confirmed, denied, voided |
| **Zero-config sandbox** | Runs with no API key — public sandbox connects automatically |

### Tuning velocity controls for your agent

```python
budget = client.create_budget(
    ...
    velocity_max_per_minute=5,        # max events per 60-second rolling window
    velocity_max_amount_per_hour=10.0, # max dollars per hour
    velocity_max_per_day=100,          # max events per calendar day
)
```

Set the limit to 2–3× your agent's normal call rate. A runaway loop always
exceeds that by an order of magnitude. Normal operation never notices the cap.

### Self-hosting

```bash
git clone https://github.com/figuard/figuard-core
cd figuard-core
docker compose up -d
```

Then set `FIGUARD_BASE_URL=http://localhost:8080` and `FIGUARD_API_KEY=<your-key>`
before importing `FiGuardClient`.